In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

df = pd.read_csv("../results/growing_partition_benchmark.csv")
df['speedup'] = df['exact_time_sec'] / df['mcmc_avg_time_sec']
df['mem_ratio'] = df['exact_peak_rss_mb'] / df['exact_peak_memory_mb'].replace(0, np.nan)

# ============================================================
# STATS
# ============================================================
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total configurations: {len(df)}")
print(f"Hidden variables range: {df['n_hidden'].min()} – {df['n_hidden'].max()}")
print(f"Partition sizes range:  {df['max_partition_size'].min()} – {df['max_partition_size'].max()}")
print(f"Exact SDP success rate: {df['exact_success'].sum()}/{len(df)}")
print(f"MCMC success rate:      {df['mcmc_success'].sum()}/{len(df)}")

print("\n" + "=" * 60)
print("TIME ANALYSIS")
print("=" * 60)
print(f"\nExact SDP time (sec):")
print(f"  min:    {df['exact_time_sec'].min():.4f}")
print(f"  max:    {df['exact_time_sec'].max():.4f}")
print(f"  median: {df['exact_time_sec'].median():.4f}")
print(f"  mean:   {df['exact_time_sec'].mean():.4f}")
print(f"\nMCMC avg time (sec):")
print(f"  min:    {df['mcmc_avg_time_sec'].min():.4f}")
print(f"  max:    {df['mcmc_avg_time_sec'].max():.4f}")
print(f"  median: {df['mcmc_avg_time_sec'].median():.4f}")
print(f"  mean:   {df['mcmc_avg_time_sec'].mean():.4f}")

crossover_rows = df[df['speedup'] >= 1.0]
if len(crossover_rows) > 0:
    crossover = crossover_rows.iloc[0]
    print(f"\nCrossover point (exact becomes slower than MCMC):")
    print(f"  n_hidden={crossover['n_hidden']} | partition_size={crossover['max_partition_size']} | speedup={crossover['speedup']:.2f}x")
print(f"\nMax speedup at partition {df['max_partition_size'].max()}: {df['speedup'].max():.1f}x")

print("\n" + "=" * 60)
print("MEMORY ANALYSIS — BOTH MEASURES")
print("=" * 60)
print(f"\nExact SDP tracemalloc (Python-tracked) peak memory (MB):")
print(f"  min:    {df['exact_peak_memory_mb'].min():.2f}")
print(f"  max:    {df['exact_peak_memory_mb'].max():.2f}")
print(f"  median: {df['exact_peak_memory_mb'].median():.2f}")
print(f"\nExact SDP RSS (OS-level) peak memory (MB):")
print(f"  min:    {df['exact_peak_rss_mb'].min():.2f}")
print(f"  max:    {df['exact_peak_rss_mb'].max():.2f}")
print(f"  median: {df['exact_peak_rss_mb'].median():.2f}")
print(f"\nMCMC peak memory (MB):")
print(f"  min:    {df['mcmc_peak_memory_mb'].min():.2f}")
print(f"  max:    {df['mcmc_peak_memory_mb'].max():.2f}")
print(f"  median: {df['mcmc_peak_memory_mb'].median():.2f}")

print("\n" + "-" * 60)
print("RSS vs tracemalloc ratio — how much tracemalloc underreports")
print("-" * 60)
for _, r in df.iterrows():
    if r['exact_peak_rss_mb'] > 0.5 and r['exact_peak_memory_mb'] > 0:
        print(f"  partition={int(r['max_partition_size']):2d} | "
              f"tracemalloc={r['exact_peak_memory_mb']:8.2f} MB | "
              f"RSS={r['exact_peak_rss_mb']:8.2f} MB | "
              f"ratio={r['mem_ratio']:.1f}×")

# Crossover: when does exact SDP (by each measure) start using more than MCMC?
tm_cross = df[df['exact_peak_memory_mb'] >= df['mcmc_peak_memory_mb']]
rss_cross = df[df['exact_peak_rss_mb'] >= df['mcmc_peak_memory_mb']]
print(f"\nMemory crossover — exact > MCMC (by tracemalloc): ", end='')
if len(tm_cross) > 0:
    r = tm_cross.iloc[0]
    print(f"partition={int(r['max_partition_size'])} | n_hidden={int(r['n_hidden'])}")
else:
    print("never")
print(f"Memory crossover — exact > MCMC (by RSS):         ", end='')
if len(rss_cross) > 0:
    r = rss_cross.iloc[0]
    print(f"partition={int(r['max_partition_size'])} | n_hidden={int(r['n_hidden'])}")
else:
    print("never")

print("\n" + "=" * 60)
print("EXPONENTIAL GROWTH ANALYSIS (EXACT SDP) — BOTH MEASURES")
print("=" * 60)
safe = df[df['exact_success'] == True].copy()

# tracemalloc fit
valid_tm = safe[safe['exact_peak_memory_mb'] > 0]
log_tm = np.log2(valid_tm['exact_peak_memory_mb'])
slope_tm, intercept_tm, r_tm, _, _ = stats.linregress(valid_tm['max_partition_size'], log_tm)
print(f"\nMemory vs partition size (tracemalloc, log2):")
print(f"  doubling rate: every {1/slope_tm:.2f} partition units (theoretical: 1.00)")
print(f"  R²: {r_tm**2:.4f}")

# RSS fit
valid_rss = safe[safe['exact_peak_rss_mb'] > 0]
log_rss = np.log2(valid_rss['exact_peak_rss_mb'])
slope_rss, intercept_rss, r_rss, _, _ = stats.linregress(valid_rss['max_partition_size'], log_rss)
print(f"\nMemory vs partition size (RSS, log2):")
print(f"  doubling rate: every {1/slope_rss:.2f} partition units")
print(f"  R²: {r_rss**2:.4f}")

# Only fit large partitions where growth is clean
large = safe[safe['max_partition_size'] >= 15]
if len(large) >= 3:
    log_tm_l = np.log2(large['exact_peak_memory_mb'])
    slope_tm_l, _, r_tm_l, _, _ = stats.linregress(large['max_partition_size'], log_tm_l)
    log_rss_l = np.log2(large[large['exact_peak_rss_mb'] > 0]['exact_peak_rss_mb'])
    prss_l = large[large['exact_peak_rss_mb'] > 0]['max_partition_size']
    slope_rss_l, _, r_rss_l, _, _ = stats.linregress(prss_l, log_rss_l)
    print(f"\nLarge-partition only fit (partition >= 15):")
    print(f"  tracemalloc: doubles every {1/slope_tm_l:.2f} units (R²={r_tm_l**2:.4f})")
    print(f"  RSS:         doubles every {1/slope_rss_l:.2f} units (R²={r_rss_l**2:.4f})")

# Time fit
log_time = np.log2(safe[safe['exact_time_sec'] > 0]['exact_time_sec'])
ptime = safe[safe['exact_time_sec'] > 0]['max_partition_size']
slope_time, intercept_time, r_time, _, _ = stats.linregress(ptime, log_time)
print(f"\nTime vs partition size (log2):")
print(f"  doubling rate: every {1/slope_time:.2f} partition units")
print(f"  R²: {r_time**2:.4f}")

print("\n" + "=" * 60)
print("MCMC ACCURACY ANALYSIS")
print("=" * 60)
accurate = df[df['mcmc_avg_estimate'] == 1.0]
inaccurate = df[df['mcmc_avg_estimate'] != 1.0]
print(f"Perfect MCMC estimate (1.0): {len(accurate)}/{len(df)}")
print(f"Inaccurate MCMC estimate:    {len(inaccurate)}/{len(df)}")
for _, row in inaccurate.iterrows():
    err = abs(1.0 - row['mcmc_avg_estimate'])
    print(f"  n_hidden={int(row['n_hidden'])} | partition={int(row['max_partition_size'])} | "
          f"estimate={row['mcmc_avg_estimate']:.4f} | abs_error={err:.4f}")

print("\n" + "=" * 60)
print("PRACTICAL THRESHOLDS — BOTH MEASURES")
print("=" * 60)
print("\nBased on tracemalloc (under-reports, optimistic):")
for mem_limit in [64, 256, 1024, 4096]:
    safe_rows = df[df['exact_peak_memory_mb'] <= mem_limit]
    if len(safe_rows) > 0:
        max_safe = safe_rows.iloc[-1]
        print(f"  Memory limit {mem_limit:5d} MB → max partition={int(max_safe['max_partition_size'])} | max n_hidden={int(max_safe['n_hidden'])}")

print("\nBased on RSS (realistic, use this for Isambard):")
for mem_limit in [64, 256, 1024, 4096, 16384]:
    safe_rows = df[df['exact_peak_rss_mb'] <= mem_limit]
    if len(safe_rows) > 0:
        max_safe = safe_rows.iloc[-1]
        print(f"  Memory limit {mem_limit:5d} MB → max partition={int(max_safe['max_partition_size'])} | max n_hidden={int(max_safe['n_hidden'])}")

print("\nBased on time:")
for time_limit in [1, 5, 30, 60]:
    safe_rows = df[df['exact_time_sec'] <= time_limit]
    if len(safe_rows) > 0:
        max_safe = safe_rows.iloc[-1]
        print(f"  Time limit   {time_limit:5d} sec → max partition={int(max_safe['max_partition_size'])} | max n_hidden={int(max_safe['n_hidden'])}")

# ============================================================
# PLOTS
# ============================================================
BLUE  = '#185FA5'
AMBER = '#B85C00'
RED   = '#A32D2D'
GREEN = '#3B6D11'
PURP  = '#534AB7'
GRAY  = '#888780'

fig = plt.figure(figsize=(14, 22))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.45, wspace=0.35)

x = df['n_hidden']

# 1. Time comparison (log)
ax1 = fig.add_subplot(gs[0, 0])
ax1.semilogy(x, df['exact_time_sec'], color=BLUE, lw=2, marker='o', ms=3, label='Exact SDP')
ax1.semilogy(x, df['mcmc_avg_time_sec'], color=AMBER, lw=2, ls='--', marker='s', ms=3, label='MCMC')
if len(crossover_rows) > 0:
    ax1.axvline(crossover_rows.iloc[0]['n_hidden'], color=GRAY, ls=':', lw=1.2,
                label=f"crossover n={int(crossover_rows.iloc[0]['n_hidden'])}")
ax1.set_xlabel('Hidden variables', fontsize=11)
ax1.set_ylabel('Time (sec, log)', fontsize=11)
ax1.set_title('Execution time: exact SDP vs MCMC', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# 2. Memory — THREE curves: tracemalloc, RSS, MCMC
ax2 = fig.add_subplot(gs[0, 1])
tm_safe = df[df['exact_peak_memory_mb'] > 0]
rss_safe = df[df['exact_peak_rss_mb'] > 0]
ax2.semilogy(tm_safe['n_hidden'], tm_safe['exact_peak_memory_mb'],
             color=BLUE, lw=2, marker='o', ms=3, label='Exact SDP (tracemalloc)')
ax2.semilogy(rss_safe['n_hidden'], rss_safe['exact_peak_rss_mb'],
             color=RED, lw=2, marker='^', ms=3, label='Exact SDP (RSS, real)')
ax2.semilogy(x, df['mcmc_peak_memory_mb'],
             color=AMBER, lw=2, ls='--', marker='s', ms=3, label='MCMC')
if len(rss_cross) > 0:
    ax2.axvline(rss_cross.iloc[0]['n_hidden'], color=GRAY, ls=':', lw=1.2,
                label=f"RSS crossover n={int(rss_cross.iloc[0]['n_hidden'])}")
ax2.set_xlabel('Hidden variables', fontsize=11)
ax2.set_ylabel('Peak memory (MB, log)', fontsize=11)
ax2.set_title('Peak memory: tracemalloc vs RSS vs MCMC', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# 3. RSS / tracemalloc ratio
ax3 = fig.add_subplot(gs[1, 0])
valid = df[(df['exact_peak_memory_mb'] > 0.1) & (df['exact_peak_rss_mb'] > 0.1)].copy()
ax3.bar(valid['n_hidden'], valid['mem_ratio'], color=PURP, width=0.7)
ax3.axhline(1.0, color=GRAY, ls='--', lw=1.2, label='parity (RSS = tracemalloc)')
ax3.set_xlabel('Hidden variables', fontsize=11)
ax3.set_ylabel('RSS / tracemalloc ratio', fontsize=11)
ax3.set_title('How much tracemalloc underreports actual memory', fontsize=12)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3, axis='y')

# 4. Speedup factor
ax4 = fig.add_subplot(gs[1, 1])
colors_bar = [PURP if s > 10 else '#7F77DD' if s > 1 else '#AFA9EC' for s in df['speedup']]
ax4.bar(x, df['speedup'], color=colors_bar, width=0.7)
ax4.axhline(1.0, color=AMBER, ls='--', lw=1.2, label='break-even (1×)')
ax4.set_xlabel('Hidden variables', fontsize=11)
ax4.set_ylabel('Speedup factor (×)', fontsize=11)
ax4.set_title('MCMC speedup over exact SDP', fontsize=12)
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3, axis='y')



# 8. Side-by-side safe-partition thresholds
ax8 = fig.add_subplot(gs[2, 0])
limits = [64, 256, 1024, 4096]
tm_maxes = []
rss_maxes = []
for lim in limits:
    tm_rows = df[df['exact_peak_memory_mb'] <= lim]
    rss_rows = df[df['exact_peak_rss_mb'] <= lim]
    tm_maxes.append(tm_rows['max_partition_size'].max() if len(tm_rows) else 0)
    rss_maxes.append(rss_rows['max_partition_size'].max() if len(rss_rows) else 0)
xx = np.arange(len(limits))
width = 0.35
ax8.bar(xx - width/2, tm_maxes, width, label='by tracemalloc', color=BLUE)
ax8.bar(xx + width/2, rss_maxes, width, label='by RSS (real)', color=RED)
ax8.set_xticks(xx)
ax8.set_xticklabels([f'{l} MB' for l in limits])
ax8.set_xlabel('Memory budget', fontsize=11)
ax8.set_ylabel('Max safe partition size', fontsize=11)
ax8.set_title('Safe thresholds — optimistic vs realistic', fontsize=12)
ax8.legend(fontsize=9)
ax8.grid(True, alpha=0.3, axis='y')
for i, (tm_v, rss_v) in enumerate(zip(tm_maxes, rss_maxes)):
    ax8.text(i - width/2, tm_v + 0.3, str(tm_v), ha='center', fontsize=9)
    ax8.text(i + width/2, rss_v + 0.3, str(rss_v), ha='center', fontsize=9)

fig.suptitle('Exact SDP vs MCMC benchmark — with tracemalloc and RSS',
             fontsize=14, fontweight='bold', y=0.995)
plt.savefig('memory_benchmark_analysis.png', dpi=150, bbox_inches='tight', facecolor='white')
print("\nPlot saved to memory_benchmark_analysis.png")

DATASET OVERVIEW
Total configurations: 20
Hidden variables range: 1 – 20
Partition sizes range:  1 – 20
Exact SDP success rate: 20/20
MCMC success rate:      20/20

TIME ANALYSIS

Exact SDP time (sec):
  min:    0.1215
  max:    23.8841
  median: 0.1619
  mean:   2.4689

MCMC avg time (sec):
  min:    1.9059
  max:    2.5004
  median: 2.0982
  mean:   2.0854

Crossover point (exact becomes slower than MCMC):
  n_hidden=17 | partition_size=17 | speedup=1.50x

Max speedup at partition 20: 11.5x

MEMORY ANALYSIS — BOTH MEASURES

Exact SDP tracemalloc (Python-tracked) peak memory (MB):
  min:    0.19
  max:    144.07
  median: 0.34

Exact SDP RSS (OS-level) peak memory (MB):
  min:    0.00
  max:    765.54
  median: 0.21

MCMC peak memory (MB):
  min:    0.66
  max:    2.28
  median: 1.30

------------------------------------------------------------
RSS vs tracemalloc ratio — how much tracemalloc underreports
------------------------------------------------------------
  partition=13 | tra